In [1]:
import pandas as pd
import numpy as np
from cmdstanpy import CmdStanModel
import geopandas as gpd
from libpysal import weights
from sklearn.preprocessing import LabelEncoder
import networkx as nx
import re
import os
import pickle
from scipy.sparse import diags

/home/yuyue/stan_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
strat_df = pd.read_csv("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/cleaned_sf.csv")

In [6]:
df = pd.read_pickle('/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/preprocessed_cleaned_all_result.pkl')

In [7]:
print(df.columns.to_list())

['id', 'time_window_start', 'tweets_struct', 'name', 'username', 'description', 'location', 'public_metrics.followers_count', 'profile_image_url', 'clean...pool', 'is_empty_window', 'window_weeks_to_election', 'profile_image_description', 'profile_image_main_info', 'Model', 'Belief_Raw', 'Belief_Cleaned', 'Demo_Raw', 'Demo_Cleaned', 'Belief_opinion_on_the_number_of_immigrants_in_america_Category', 'Belief_opinion_on_the_number_of_immigrants_in_america_Explanation', 'Belief_opinion_on_the_number_of_immigrants_in_america_Symbol', 'Belief_opinion_on_the_number_of_immigrants_in_america_Speculation', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Category', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Explanation', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Symbol', 'Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Speculation', 'Belief_opinion_on_the_statement_immigrants_take_jobs_away_from_people_who_were_born_in_t

In [8]:
belief_col = "Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol"

predictor_cols = [
    "Demo_sex_Symbol",
    "Demo_age_Symbol",
    "Demo_ethnicity_Symbol",
    "Demo_income_Symbol",
    "Demo_past_vote_Symbol",
    "state_cleaned"
]
df = df.dropna(subset=[belief_col] + predictor_cols)

In [9]:
df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,...,Belief_opinion_on_whether_the_tone_of_political_discourse_in_the_u_s_has_become_more_factbased_Symbol_num,Belief_opinion_on_the_statement_elected_officials_using_heated_or_aggressive_language_to_talk_about_certain_people_or_groups_makes_violence_against_those_groups_more_likely_Symbol_cat,Belief_opinion_on_the_statement_elected_officials_using_heated_or_aggressive_language_to_talk_about_certain_people_or_groups_makes_violence_against_those_groups_more_likely_Symbol_num,Demo_sex_idx,Demo_age_idx,Demo_ethnicity_idx,Demo_income_idx,Demo_education_idx,Demo_past_vote_idx,state_cleaned_idx
40,1542629834953719810,2024-05-19 00:00:00+00:00,"['24 weeks to election', {'text': '@The_Trump_...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,2,dib1,1,2.0,3.0,1.0,3.0,1.0,2.0,36.0
41,1542629834953719810,2024-06-16 00:00:00+00:00,"['20 weeks to election', {'text': '🤣 https://t...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,2,dib1,1,2.0,3.0,1.0,3.0,1.0,2.0,36.0
43,1542629834953719810,2024-07-14 00:00:00+00:00,"['16 weeks to election', {'text': '🤣🤣🤣 https:/...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,2,dib3,3,2.0,3.0,1.0,3.0,1.0,2.0,36.0
44,1542629834953719810,2024-07-28 00:00:00+00:00,"['14 weeks to election', {'text': '@victormarx...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,2,dib1,1,2.0,3.0,1.0,2.0,1.0,3.0,36.0
45,1542629834953719810,2024-08-11 00:00:00+00:00,"['12 weeks to election', {'text': '🤣🥰 https://...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,2,dib1,1,2.0,3.0,1.0,3.0,1.0,2.0,36.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23984,142827500,2024-08-11 00:00:00+00:00,"['12 weeks to election', {'text': '@maxies_bac...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,dib1,1,1.0,2.0,1.0,4.0,2.0,3.0,15.0
23985,142827500,2024-08-25 00:00:00+00:00,"['10 weeks to election', {'text': '@foldyrhand...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,dib1,1,1.0,2.0,1.0,3.0,2.0,3.0,15.0
23987,142827500,2024-09-22 00:00:00+00:00,"['6 weeks to election', {'text': '@kevin__fm S...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,dib1,1,1.0,2.0,1.0,3.0,2.0,3.0,15.0
23988,142827500,2024-10-06 00:00:00+00:00,"['4 weeks to election', {'text': '@cherryrepre...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,2,dib1,1,1.0,2.0,1.0,3.0,2.0,1.0,15.0


In [11]:
# prepare spatial structure for ICAR
def edges_from_adjacency(W):
    all_edges = []
    n = W.shape[0]
    
    for i in range(n):
        neighbors = np.where(W[i, :] != 0)[0]
        for j in neighbors:
            if i < j:  
                all_edges.append({
                    'node1': i + 1,  
                    'node2': j + 1,
                    'weight': W[i, j]
                })
    
    return pd.DataFrame(all_edges)


def scale_c(C):
    N = C.shape[0]
    
    if hasattr(C, 'toarray'):
        C = C.toarray()
    C = np.asarray(C)
    
    row_sums = C.sum(axis=1).ravel()
    Q = diags(row_sums) - C
    
    Q_dense = Q.toarray() if hasattr(Q, 'toarray') else np.asarray(Q)
    max_diag = np.max(np.diag(Q_dense))
    jitter = max_diag * np.sqrt(np.finfo(float).eps)
    Q_pert = Q_dense + np.diag(np.ones(N) * jitter)
    
    Q_inv = np.linalg.pinv(Q_pert)
    
    variances = np.diag(Q_inv)
    scaling_factor = np.exp(np.mean(np.log(variances)))
    
    return scaling_factor


def prep_icar_data_from_weights(w, N_states):
    C = np.zeros((N_states, N_states))
    for i in range(N_states):
        for j in w.neighbors[i]:
            C[i, j] = 1

    E = edges_from_adjacency(C)
    
    G = nx.Graph()
    G.add_nodes_from(range(N_states))
    for _, row in E.iterrows():
        G.add_edge(int(row['node1']) - 1, int(row['node2']) - 1)
    
    components = list(nx.connected_components(G))
    k = len(components)
    
    comp_id = np.zeros(N_states, dtype=int)
    for idx, component in enumerate(components):
        for node in component:
            comp_id[int(node)] = idx
    
    group_size = np.array([len(comp) for comp in components])
    
    group_idx = comp_id + 1
    
    scale_factor = scale_c(C)
    inv_sqrt_scale_factor = np.ones(k) / np.sqrt(scale_factor)
    
    return {
        'k': int(k),
        'group_size': group_size.tolist(),
        'N_edges': len(E),
        'node1': E['node1'].values.tolist(),
        'node2': E['node2'].values.tolist(),
        'group_idx': group_idx.tolist(),
        'inv_sqrt_scale_factor': inv_sqrt_scale_factor.tolist(),
    }


def moran_coefficient(x, W, warn=True):
    row_sums = W.sum(axis=1)
    zero_idx = np.where(row_sums == 0)[0]
    
    if len(zero_idx) > 0:
        if warn:
            print(f"{len(zero_idx)} observations with no neighbors. Dropping from calculation.")
        keep_idx = row_sums > 0
        x = x[keep_idx]
        W = W[np.ix_(keep_idx, keep_idx)]
    
    n = len(x)
    xbar = np.mean(x)
    z = x - xbar
    z_tilde = W @ z
    A = np.sum(W)
    mc = (n / A) * (z @ z_tilde) / (z @ z)
    
    return np.round(mc, 3)

# Build spatial structure
shp = gpd.read_file("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/tl_2025_us_state/tl_2025_us_state.shp")
shp["STATE"] = shp["STUSPS"].str.upper()

shp = shp[~shp["STATE"].isin(["PR", "GU", "VI", "MP", "AS"])]

shp = shp.sort_values("STATE").reset_index(drop=True)

state_order = shp["STATE"].tolist()
N_states = len(state_order)

print(f"Shapefile states: {N_states}")

w = weights.Queen.from_dataframe(shp)

icar_data = prep_icar_data_from_weights(w, N_states)

print(f"\nICAR structure:")
print(f"  Connected components: {icar_data['k']}")
print(f"  Component sizes: {icar_data['group_size']}")
print(f"  Number of edges: {icar_data['N_edges']}")
print(f"  Scaling factors: {icar_data['inv_sqrt_scale_factor']}")

# Check spatial autocorrelation in outcome
C = np.zeros((N_states, N_states))
for i in range(N_states):
    for j in w.neighbors[i]:
        C[i, j] = 1


# Aggregate outcome by state for Moran's I
state_means = np.full(N_states, np.nan)

df['state_cleaned_idx'] = df['state_cleaned_idx'].astype(int)
grouped = df.groupby("state_cleaned_idx")[belief_col+"_num"].mean()
for idx, val in grouped.items():
    state_means[idx - 1] = val

mc_val = moran_coefficient(state_means, C)


Shapefile states: 51


/tmp/ipykernel_696158/674881532.py:114: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = weights.Queen.from_dataframe(shp)



ICAR structure:
  Connected components: 3
  Component sizes: [1, 49, 1]
  Number of edges: 112
  Scaling factors: [0.0022392947909694485, 0.0022392947909694485, 0.0022392947909694485]
2 observations with no neighbors. Dropping from calculation.


/home/yuyue/stan_env/lib/python3.12/site-packages/libpysal/weights/contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
 There are 2 islands with ids: 0, 11.
  W.__init__(self, neighbors, ids=ids, **kw)


In [12]:
shp.head()

,REGION,DIVISION,STATEFP,STATENS,GEOID,GEOIDFQ,STUSPS,NAME,LSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,STATE
0,4,9,02,01785533,02,0400000US02,AK,Alaska,00,G4000,A,1479893380150,244326118163,+63.3473560,-152.8397334,"MULTIPOLYGON (((-167.55823 60.22436, -167.5567...",AK
1,3,6,01,01779775,01,0400000US01,AL,Alabama,00,G4000,A,131186429591,4580729056,+32.7395785,-086.8434469,"POLYGON ((-85.12733 31.76256, -85.12753 31.762...",AL
2,3,7,05,00068085,05,0400000US05,AR,Arkansas,00,G4000,A,134658642291,3122591253,+34.8955256,-092.4446262,"POLYGON ((-90.95577 34.11871, -90.95451 34.117...",AR
3,4,8,04,01779777,04,0400000US04,AZ,Arizona,00,G4000,A,294366238828,853868947,+34.2039362,-111.6063449,"POLYGON ((-110.75069 37.00301, -110.74193 37.0...",AZ
4,4,9,06,01779778,06,0400000US06,CA,California,00,G4000,A,403676564152,20288505023,+37.1551773,-119.5434183,"MULTIPOLYGON (((-119.99987 41.18397, -119.9998...",CA


In [13]:
# state level fixed effects
strat_frame = pd.read_csv("/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/feature_extraction/cleaned_sf.csv")
LP_cols = [
    'LP_pct.2020.votes.R', 'LP_pct.2020.votes.stay.home'
]
LP_state = (
    strat_frame
    .assign(STATE=strat_frame["state_abbreviation"].str.upper())
    .groupby("STATE")[LP_cols]
    .mean()
)
LP_state = LP_state.reindex(state_order)

LP_mat = LP_state.values
P = LP_mat.shape[1]
print(LP_mat.shape)

(51, 2)


In [14]:
print(LP_mat)

[[ 0.3032589  -0.15494699]
 [ 1.06868261  0.92695691]
 [ 1.09897892  1.87393581]
 [-0.01104556 -0.13549883]
 [-1.23719282 -0.06126084]
 [-0.63112784 -1.72932917]
 [-0.83225137 -0.36399364]
 [-3.64396112  0.30521813]
 [-0.78334293 -0.39551991]
 [ 0.16901134 -0.80350951]
 [ 0.00405679 -0.12421476]
 [-1.26764912  1.97166449]
 [ 0.30051743 -0.98120931]
 [ 1.2189566   0.05901784]
 [-0.71855639  0.2087094 ]
 [ 0.65175111  1.08771718]
 [ 0.58446232  0.28206992]
 [ 1.07325333  0.47984469]
 [ 0.77155319  0.61767264]
 [-1.43794678 -0.65951938]
 [-1.41779837 -0.34927793]
 [-0.41779583 -1.474638  ]
 [-0.11245498 -1.00240798]
 [-0.32484046 -2.09808463]
 [ 0.63331954  0.20469241]
 [ 0.70016347  1.25593282]
 [ 0.64320622 -0.79548343]
 [ 0.06204364 -0.91829113]
 [ 1.32517512  0.71970753]
 [ 0.83397673 -0.2725791 ]
 [-0.31890604 -1.20076828]
 [-0.64840276 -0.5917758 ]
 [-0.47364589  1.10262997]
 [-0.1266878   0.19441404]
 [-0.97586033  1.07755046]
 [ 0.33971872  0.12281927]
 [ 1.34674554  2.06296573]
 

In [15]:
LP_df = pd.DataFrame(
    LP_mat,
    index=[s.upper() for s in state_order],
    columns=LP_cols
)

LP_df

,LP_pct.2020.votes.R,LP_pct.2020.votes.stay.home
AK,0.303259,-0.154947
AL,1.068683,0.926957
AR,1.098979,1.873936
AZ,-0.011046,-0.135499
CA,-1.237193,-0.061261
CO,-0.631128,-1.729329
CT,-0.832251,-0.363994
DC,-3.643961,0.305218
DE,-0.783343,-0.395520
FL,0.169011,-0.803510


In [16]:
print(df["Demo_age_idx"].unique())

[3. 4. 5. 2. 6. 1.]


In [17]:
# Stan data
stan_data = {
    "N": len(df),
    "K": df[belief_col+"_cat"].cat.categories.size,
    "y": df[belief_col+"_num"].astype(int).tolist(),
    
    # Predictor dimensions
    "J_sex": strat_df['sex_idx'].nunique(),
    "J_age": strat_df['age_idx'].nunique(),
    "J_eth": strat_df['eth_idx'].nunique(),
    "J_income": strat_df['income_idx'].nunique(),
    "J_vote": strat_df['vote_idx'].nunique(),
    "N_states": N_states,
    

    # Predictor indices
    "sex_idx": df['Demo_sex_idx'].astype(int).tolist(),
    "age_idx": df['Demo_age_idx'].astype(int).tolist(),
    "eth_idx": df['Demo_ethnicity_idx'].astype(int).tolist(),
    "income_idx": df['Demo_income_idx'].astype(int).tolist(),
    "vote_idx": df['Demo_past_vote_idx'].astype(int).tolist(),
    "state_idx": df['state_cleaned_idx'].astype(int).tolist(),

    # ICAR spatial structure
    **icar_data,

    # state level fix effect
    "P": LP_mat.shape[1],
    "LP": LP_mat.tolist(),
}

print(f"\nStan data prepared with {len(stan_data)} components")



Stan data prepared with 24 components


In [18]:
print("inv_sqrt_scale_factor values:", icar_data['inv_sqrt_scale_factor'])
print("Number of components:", icar_data['k'])
print("Component sizes:", icar_data['group_size'])

inv_sqrt_scale_factor values: [0.0022392947909694485, 0.0022392947909694485, 0.0022392947909694485]
Number of components: 3
Component sizes: [1, 49, 1]


In [19]:
shp_states = shp["STATE"].tolist()

print("Shapefile states:", shp_states)
print("Match with state_order:", shp_states == [s.upper() for s in state_order])


Shapefile states: ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']
Match with state_order: True


In [20]:
assert set(shp_states) == set(s.upper() for s in state_order), \
    "Shapefile and sample states do not match"


In [21]:
idx_to_state = {i+1: s.upper() for i, s in enumerate(state_order)}

df["state_from_idx"] = df["state_cleaned_idx"].map(idx_to_state)


In [22]:
mismatch = df[df["state_from_idx"] != df["state_cleaned"].str.upper()]
print("Number of mismatches:", len(mismatch))


Number of mismatches: 0


In [23]:
df

,id,time_window_start,tweets_struct,name,username,description,location,public_metrics.followers_count,profile_image_url,clean...pool,...,Belief_opinion_on_the_statement_elected_officials_using_heated_or_aggressive_language_to_talk_about_certain_people_or_groups_makes_violence_against_those_groups_more_likely_Symbol_cat,Belief_opinion_on_the_statement_elected_officials_using_heated_or_aggressive_language_to_talk_about_certain_people_or_groups_makes_violence_against_those_groups_more_likely_Symbol_num,Demo_sex_idx,Demo_age_idx,Demo_ethnicity_idx,Demo_income_idx,Demo_education_idx,Demo_past_vote_idx,state_cleaned_idx,state_from_idx
40,1542629834953719810,2024-05-19 00:00:00+00:00,"['24 weeks to election', {'text': '@The_Trump_...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,dib1,1,2.0,3.0,1.0,3.0,1.0,2.0,36,OH
41,1542629834953719810,2024-06-16 00:00:00+00:00,"['20 weeks to election', {'text': '🤣 https://t...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,dib1,1,2.0,3.0,1.0,3.0,1.0,2.0,36,OH
43,1542629834953719810,2024-07-14 00:00:00+00:00,"['16 weeks to election', {'text': '🤣🤣🤣 https:/...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,dib3,3,2.0,3.0,1.0,3.0,1.0,2.0,36,OH
44,1542629834953719810,2024-07-28 00:00:00+00:00,"['14 weeks to election', {'text': '@victormarx...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,dib1,1,2.0,3.0,1.0,2.0,1.0,3.0,36,OH
45,1542629834953719810,2024-08-11 00:00:00+00:00,"['12 weeks to election', {'text': '🤣🥰 https://...",Ashley Diller,Badash1821,A mom of 5 with common sense! Hates being lie...,"Ohio, USA",625,https://pbs.twimg.com/profile_images/169368304...,radical,...,dib1,1,2.0,3.0,1.0,3.0,1.0,2.0,36,OH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23984,142827500,2024-08-11 00:00:00+00:00,"['12 weeks to election', {'text': '@maxies_bac...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,dib1,1,1.0,2.0,1.0,4.0,2.0,3.0,15,IL
23985,142827500,2024-08-25 00:00:00+00:00,"['10 weeks to election', {'text': '@foldyrhand...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,dib1,1,1.0,2.0,1.0,3.0,2.0,3.0,15,IL
23987,142827500,2024-09-22 00:00:00+00:00,"['6 weeks to election', {'text': '@kevin__fm S...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,dib1,1,1.0,2.0,1.0,3.0,2.0,3.0,15,IL
23988,142827500,2024-10-06 00:00:00+00:00,"['4 weeks to election', {'text': '@cherryrepre...",Andy,ndrwwwww,architectural designer \n\nhe/him,"Chicago, IL",734,https://pbs.twimg.com/profile_images/137092184...,guns,...,dib1,1,1.0,2.0,1.0,3.0,2.0,1.0,15,IL


In [24]:
dem_cols = ["Demo_sex_idx", "Demo_age_idx", "Demo_ethnicity_idx", "Demo_income_idx", "Demo_past_vote_idx"]

# Map raw column names to strat frame keys
dem_mapping_keys = {
    "Demo_sex_idx": "sex_idx",
    "Demo_age_idx": "age_idx",
    "Demo_ethnicity_idx": "eth_idx",
    "Demo_income_idx": "income_idx",
    "Demo_past_vote_idx": "vote_idx"
}

dem_mapping = {
    "sex_idx": strat_df[['sex_idx']].drop_duplicates().sort_values('sex_idx'),
    "age_idx": strat_df[['age_idx']].drop_duplicates().sort_values('age_idx'),
    "eth_idx": strat_df[['eth_idx']].drop_duplicates().sort_values('eth_idx'),
    "income_idx": strat_df[['income_idx']].drop_duplicates().sort_values('income_idx'),
    "vote_idx": strat_df[['vote_idx']].drop_duplicates().sort_values('vote_idx'),
}

for col in dem_cols:
    strat_key = dem_mapping_keys[col]  # map raw col to strat frame
    raw_unique = df[col].dropna().unique()
    strat_unique = dem_mapping[strat_key][strat_key].values
    print(f"{col}:")
    print(f"  Raw data unique values: {sorted(raw_unique)}")
    print(f"  Strat frame values: {sorted(strat_unique)}")
    if set(raw_unique) <= set(strat_unique):
        print("  All raw data indices exist in strat frame")
    else:
        print("  Some indices in raw data are missing in strat frame")
    print("------------")


Demo_sex_idx:
  Raw data unique values: [np.float64(1.0), np.float64(2.0)]
  Strat frame values: [np.int64(1), np.int64(2)]
  All raw data indices exist in strat frame
------------
Demo_age_idx:
  Raw data unique values: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0)]
  Strat frame values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
  All raw data indices exist in strat frame
------------
Demo_ethnicity_idx:
  Raw data unique values: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0)]
  Strat frame values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
  All raw data indices exist in strat frame
------------
Demo_income_idx:
  Raw data unique values: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
  Strat frame values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int6

In [28]:
# Stan code
stan_code = '''
functions {
  real icar_normal_lpdf(vector phi, int N_edges,
                        array[] int node1, array[] int node2) {
    return -0.5 * dot_self(phi[node1] - phi[node2]);
  }
}

data {
  int<lower=1> N;                   
  int<lower=2> K;                 
  array[N] int<lower=1, upper=K> y;

  int<lower=1> J_sex;
  int<lower=1> J_age;
  int<lower=1> J_eth;
  int<lower=1> J_income;
  int<lower=1> J_vote;
  int<lower=1> N_states;

  array[N] int<lower=1, upper=J_sex> sex_idx;
  array[N] int<lower=1, upper=J_age> age_idx;
  array[N] int<lower=1, upper=J_eth> eth_idx;
  array[N] int<lower=1, upper=J_income> income_idx;
  array[N] int<lower=1, upper=J_vote> vote_idx;
  array[N] int<lower=1, upper=N_states> state_idx;

  // ICAR structure
  int<lower=1> N_edges;
  array[N_edges] int node1;
  array[N_edges] int node2;

  int<lower=1> k;                           // number of connected components
  array[k] int group_size;                  // size of each connected component
  array[N_states] int group_idx;           // component assignment for each state

  // state level fixed effects
  int<lower=1> P;                           // number of state-level predictors
  matrix[N_states, P] LP;                  // state-level predictor matrix
}

parameters {
  ordered[K-1] kappa;

  sum_to_zero_vector[J_sex] eta_sex; // use build in sum_to_zero_vector instead of hard coding it
  sum_to_zero_vector[J_age] eta_age;             
  sum_to_zero_vector[J_eth] eta_eth;
  sum_to_zero_vector[J_income] eta_income;      
  sum_to_zero_vector[J_vote] eta_vote;
  vector[N_states] eta_state_raw;
  vector[P] beta_LP;

  real<lower=0> sigma_sex;
  real<lower=0> sigma_age;
  real<lower=0> sigma_eth;
  real<lower=0> sigma_income;
  real<lower=0> sigma_vote;
  real<lower=0> sigma_state;
}

transformed parameters {
  vector[J_sex] eta_sex_star = eta_sex * sigma_sex * sqrt((J_sex / (J_sex - 1.0)));
  vector[J_age] eta_age_star = eta_age * sigma_age;
  vector[J_eth] eta_eth_star = eta_eth * sigma_eth * sqrt((J_eth / (J_eth - 1.0)));
  vector[J_income] eta_income_star = eta_income * sigma_income;
  vector[J_vote] eta_vote_star = eta_vote * sigma_vote * sqrt((J_vote / (J_vote - 1.0)));
  vector[N_states] mu_state_LP = LP * beta_LP; 
  vector[N_states] eta_state = sigma_state * eta_state_raw;
}

model {
  // Priors
  kappa ~ normal(0, 3);
  beta_LP ~ normal(0, 1);

   // prior on the constrained vector (Stan handles the internal K-1 representation)
  eta_sex ~ normal(0, 1);
  eta_eth ~ normal(0, 1);
  eta_vote ~ normal(0, 1);
  
  sigma_sex ~ normal(0, 1);
  sigma_age ~ normal(0, 1);
  sigma_eth ~ normal(0, 1);
  sigma_income ~ normal(0, 1);
  sigma_vote ~ normal(0, 1);
  sigma_state ~ normal(0, 1);

  for (i in 2:J_age) {
    eta_age[i] ~ normal(eta_age[i-1], 1);
  }
  for (i in 2:J_income) {
    eta_income[i] ~ normal(eta_income[i-1], 1);
  }

  // ICAR prior on raw spatial effects
  target += icar_normal_lpdf(eta_state_raw | N_edges, node1, node2);
  
  // Sum-to-zero constraint per connected component (island can have free prior)
  for (j in 1:k) {
  if (group_size[j] > 1) {
    // Soft sum-to-zero constraint for proper ICAR within this component
    real component_sum = 0;
    for (i in 1:N_states) {
      if (group_idx[i] == j) {
        component_sum += eta_state_raw[i];
      }
    }
    target += normal_lpdf(component_sum | 0, 0.001 * group_size[j]);
  } else {
    // Island component: give the singleton node a proper iid prior
    for (i in 1:N_states) {
      if (group_idx[i] == j) {
        eta_state_raw[i] ~ normal(0, 1);
      }
    }
  }
}

  // Ordered logistic
  for (i in 1:N) {
    real mu = eta_sex_star[sex_idx[i]]
              + eta_age_star[age_idx[i]]
              + eta_eth_star[eth_idx[i]]
              + eta_income_star[income_idx[i]]
              + eta_vote_star[vote_idx[i]]
              + eta_state[state_idx[i]]
              + mu_state_LP[state_idx[i]];
    y[i] ~ ordered_logistic(mu, kappa);
  }
}

generated quantities {
  vector[N] mu;
  vector[N] log_lik;
  
  for (i in 1:N) {
    mu[i] = eta_sex_star[sex_idx[i]]
            + eta_age_star[age_idx[i]]
            + eta_eth_star[eth_idx[i]]
            + eta_income_star[income_idx[i]]
            + eta_vote_star[vote_idx[i]]
            + eta_state[state_idx[i]]
            + mu_state_LP[state_idx[i]];
    log_lik[i] = ordered_logistic_lpmf(y[i] | mu[i], kappa);
  }
}
'''


# Compile and Sample

with open('mrp.stan', 'w') as f:
    f.write(stan_code)

model = CmdStanModel(stan_file='mrp.stan')

fit = model.sample(
    data=stan_data,
    chains=1,
    parallel_chains=1,
    iter_warmup=100,
    iter_sampling=50,
    thin=10,
    adapt_delta=0.8,
    max_treedepth=8,
    sig_figs=18,
    show_console=True
)

print(fit.summary())

base_dir = "/mnt/e/Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/test_for_storage_Dropped_ICAR_MrP"
model_dir = f"{base_dir}/{belief_col}_stan_output"

os.makedirs(model_dir, exist_ok=True)

fit.save_csvfiles(dir=f"{model_dir}/mrp_results/")
summary_df = fit.summary()

summary_df.to_csv(f"{model_dir}/mrp_summary.csv", index=True)

posterior_draws = fit.draws_pd()  

posterior_draws.to_csv(f"{model_dir}/mrp_draws.csv", index=False)
with open(f"{model_dir}/fit.pkl", "wb") as f:
    pickle.dump(fit, f)

21:31:13 - cmdstanpy - INFO - compiling stan file /tmp/tmpfqc1oa61/tmpalwyp2lo.stan to exe file /mnt/c/Users/yuyue/AppData/Local/Programs/Microsoft VS Code/mrp
21:31:30 - cmdstanpy - INFO - compiled model executable: /mnt/c/Users/yuyue/AppData/Local/Programs/Microsoft VS Code/mrp
21:31:30 - cmdstanpy - INFO - Chain [1] start processing


Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 50
Chain [1] num_warmup = 100
Chain [1] save_warmup = false (Default)
Chain [1] thin = 10
Chain [1] adapt
Chain [1] engaged = true (Default)
Chain [1] gamma = 0.05 (Default)
Chain [1] delta = 0.8 (Default)
Chain [1] kappa = 0.75 (Default)
Chain [1] t0 = 10 (Default)
Chain [1] init_buffer = 75 (Default)
Chain [1] term_buffer = 50 (Default)
Chain [1] window = 25 (Default)
Chain [1] save_metric = false (Default)
Chain [1] algorithm = hmc (Default)
Chain [1] hmc
Chain [1] engine = nuts (Default)
Chain [1] nuts
Chain [1] max_depth = 8
Chain [1] metric = diag_e (Default)
Chain [1] metric_file =  (Default)
Chain [1] stepsize = 1 (Default)
Chain [1] stepsize_jitter = 0 (Default)
Chain [1] num_chains = 1 (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /tmp/tmpt_jksh4v/_z7ej460.json
Chain [1] init = 2 (Default)
Chain [1] random
Chain [1] seed = 82484
Chain [1] output
Chain [1] file = /tmp/tmpt_jksh4

21:32:29 - cmdstanpy - INFO - Chain [1] done processing
21:32:29 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: ordered_logistic: Final cut-point is inf, but must be finite! (in 'mrp.stan', line 128, column 4 to column 39)
	Exception: ordered_logistic: Final cut-point is inf, but must be finite! (in 'mrp.stan', line 128, column 4 to column 39)
	Exception: ordered_logistic: Final cut-point is inf, but must be finite! (in 'mrp.stan', line 128, column 4 to column 39)
	Exception: ordered_logistic: Final cut-point is inf, but must be finite! (in 'mrp.stan', line 128, column 4 to column 39)
21:32:29 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 4 iterations at max treedepth (8.0%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


Chain [1] 
Chain [1] Elapsed Time: 36.032 seconds (Warm-up)
Chain [1] 22.91 seconds (Sampling)
Chain [1] 58.942 seconds (Total)
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Chain [1] 
Cha

In [29]:
print(f'draws as array:  {fit.draws().shape}')

draws as array:  (5, 1, 14932)


In [21]:
df.select_dtypes(include=["number"]).isna().sum()

id                                                                             0
public_metrics.followers_count                                                 0
window_weeks_to_election                                                       0
Belief_opinion_on_the_number_of_immigrants_in_america_Speculation              0
Belief_opinion_on_the_statement_immigrants_increase_crime_rates_Speculation    0
                                                                              ..
Demo_age_idx                                                                   0
Demo_ethnicity_idx                                                             0
Demo_income_idx                                                                0
Demo_past_vote_idx                                                             0
state_cleaned_idx                                                              0
Length: 103, dtype: int64

In [22]:
df.select_dtypes(include=["number"]).abs().max().sort_values()


Demo_sex_idx                                                                                                                  2.000000e+00
Belief_opinion_on_whether_the_us_has_a_responsibility_to_help_ukraine_defend_itself_from_russia_s_invasion_Symbol_num         2.000000e+00
Belief_opinion_on_what_should_be_done_with_illegal_immigrants_Symbol_num                                                      3.000000e+00
Belief_opinion_on_whether_divorce_in_the_usa_should_be_easier_or_more_difficult_to_obtain_than_it_is_Symbol_num               3.000000e+00
Belief_opinion_on_expanding_the_child_tax_credit_Symbol_num                                                                   3.000000e+00
                                                                                                                                  ...     
Belief_opinion_on_whether_divorce_in_the_usa_should_be_easier_or_more_difficult_to_obtain_than_it_is_Speculation              1.000000e+02
Belief_opinion_on_the_follo

In [23]:
df[f"{belief_col}_num"].value_counts()


Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_num
1    4277
2    2680
3     399
Name: count, dtype: int64

In [24]:
for group in ["Demo_sex_idx", "Demo_age_idx", "Demo_ethnicity_idx", 
              "Demo_income_idx", "Demo_past_vote_idx", "state_cleaned_idx"]:
    print("\nChecking", group)
    print(df.groupby(group)["Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_num"]
            .nunique())


Checking Demo_sex_idx
Demo_sex_idx
1.0    3
2.0    3
Name: Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_num, dtype: int64

Checking Demo_age_idx
Demo_age_idx
1.0    3
2.0    3
3.0    3
4.0    3
5.0    3
6.0    3
Name: Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_num, dtype: int64

Checking Demo_ethnicity_idx
Demo_ethnicity_idx
1.0    3
2.0    3
3.0    3
4.0    3
5.0    2
6.0    3
Name: Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_num, dtype: int64

Checking Demo_income_idx
Demo_income_idx
1.0    3
2.0    3
3.0    3
4.0    3
5.0    3
Name: Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_num, dtype: int64

Checking Demo_past_vote_idx
Demo_past_vote_idx
1.0    3
2.0    3
3.0    3
4.0    3
5.0    1
6.0    3
Name: Belief_opinion_on_january_6th_storming_of_the_us_capitol_Symbol_num, dtype: int64

Checking state_cleaned_idx
state_cleaned_idx
2     3
3     3
4     3
5     3
6     3
7     3
8     3
9     2
10    3
11    3